# Auditoría de memoria GPU por arquitectura y batch (sin entrenamiento)Mide DeepLabV3+ semántico y Mask R-CNN de instancias a 512×512, con batch 1, 2 y 4. Cada combinación se ejecuta en un proceso aislado para contener errores OOM. Usa tensores sintéticos: **no descarga ni lee datasets, no toca test, no guarda checkpoints y nunca llama a optimizer.step()**. Sí realiza un forward y un backward para medir memoria de activaciones de forma representativa.El resultado orienta el batch inicial; no demuestra estabilidad de una época completa ni autoriza entrenamiento. La memoria real puede variar con imágenes, número de objetos, augmentations, workers y validación.

## 1. Montar Google Drive y validar el preflight aprobadoLos informes se guardarán en `MyDrive/ProcesamientoImagenes/auditoria_memoria_gpu`.

In [ ]:
from google.colab import drivefrom pathlib import Pathimport os, sys, json, subprocess, time, base64import pandas as pddrive.mount('/content/drive')OUT = Path('/content/drive/MyDrive/ProcesamientoImagenes/auditoria_memoria_gpu')OUT.mkdir(parents=True, exist_ok=True)PREFLIGHT = Path('/content/drive/MyDrive/ProcesamientoImagenes/auditoria_entorno_entrenamiento/training_runtime_preflight.json')if not PREFLIGHT.is_file():    raise RuntimeError('No encuentro training_runtime_preflight.json. Ejecuta primero el preflight de entorno.')pre = json.loads(PREFLIGHT.read_text())if pre.get('status') != 'PASS' or pre.get('gpu_status') != 'PASS' or pre.get('drive_status') != 'PASS':    raise RuntimeError('El preflight GPU/Drive no está aprobado; detengo la auditoría.')print('Preflight aprobado:', pre.get('torch'), pre.get('devices', [{}])[0].get('name'))

## 2. Sincronizar el repositorio privadoUsa `GITHUB_TOKEN` en Colab Secrets (Contents: Read-only). No pegues el token en el notebook ni lo compartas.

In [ ]:
from getpass import getpassREPO = 'Leohf04/Procesamiento-de-Im-genes-'BRANCH = 'urban-models-and-benchmark'WORKDIR = Path('/content/Procesamiento-de-Im-genes-')try:    from google.colab import userdata    TOKEN = userdata.get('GITHUB_TOKEN')except Exception:    TOKEN = Noneif not TOKEN:    TOKEN = getpass('GitHub PAT (Contents: Read-only): ')if not TOKEN:    raise RuntimeError('Falta GITHUB_TOKEN')# Autenticación temporal por header HTTP; el token no queda en la URL ni en el repo.auth = base64.b64encode(('x-access-token:' + TOKEN).encode()).decode()env = os.environ.copy()env['GIT_CONFIG_COUNT'] = '1'env['GIT_CONFIG_KEY_0'] = 'http.extraheader'env['GIT_CONFIG_VALUE_0'] = 'AUTHORIZATION: basic ' + authenv['GIT_TERMINAL_PROMPT'] = '0'if WORKDIR.exists():    subprocess.run(['git', '-C', str(WORKDIR), 'fetch', 'origin', BRANCH], check=True, env=env)    subprocess.run(['git', '-C', str(WORKDIR), 'checkout', BRANCH], check=True)    subprocess.run(['git', '-C', str(WORKDIR), 'pull', '--ff-only', 'origin', BRANCH], check=True, env=env)else:    subprocess.run(['git', 'clone', '--single-branch', '--branch', BRANCH,                    f'https://github.com/{REPO}.git', str(WORKDIR)], check=True, env=env)del TOKEN, auth, envos.chdir(WORKDIR)HEAD = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()print('HEAD:', HEAD)if not Path('scripts/audit_model_memory.py').is_file():    raise RuntimeError('Falta scripts/audit_model_memory.py en el HEAD clonado.')

## 3. Ejecutar combinaciones aisladasCada proceso instancia el modelo real de la arquitectura, usa datos sintéticos 512×512, ejecuta forward/backward y termina sin optimizar pesos. Orden: batch 1, 2 y 4 por tarea. Si una combinación da OOM, el proceso termina y la siguiente se intenta en un proceso nuevo. Se mide AMP desactivado para coincidir con la configuración conservadora actual del protocolo.

In [ ]:
CASES = [(task, batch) for task in ('semantic', 'instances') for batch in (1, 2, 4)]rows = []for task, batch in CASES:    out = OUT / f'{task}_b{batch}_512.json'    cmd = [sys.executable, 'scripts/audit_model_memory.py', '--task', task,           '--batch-size', str(batch), '--image-size', '512', '--output', str(out)]    print('\n' + '=' * 72 + f'\n{task} batch={batch}\n' + '=' * 72)    proc = subprocess.run(cmd, cwd=WORKDIR, text=True, capture_output=True)    print(proc.stdout)    if proc.stderr:        print('STDERR:', proc.stderr[-3000:])    if not out.is_file():        raise RuntimeError(f'No se generó el informe para {task}/batch={batch}')    item = json.loads(out.read_text())    item['process_exit_code'] = proc.returncode    rows.append(item)    if item.get('status') == 'FAIL':        raise RuntimeError(f"Error no-OOM en {task}/batch={batch}: {item.get('error')}")

## 4. Resumen y criterio de selecciónNo se debe escoger batch 4 por ser el mayor que cabe en una única pasada. Para el piloto se prioriza batch 1 si es estable; se aumenta solo si deja margen suficiente de VRAM. Mask R-CNN es especialmente sensible al número de propuestas/objetos y estas muestras sintéticas no cubren el peor caso.

In [ ]:
df = pd.DataFrame([{k: r.get(k) for k in (    'task', 'batch_size', 'image_size', 'status', 'gpu',    'peak_allocated_bytes', 'peak_reserved_bytes',    'peak_increment_over_model_bytes', 'elapsed_seconds',    'backward_calls', 'optimizer_steps')} for r in rows])for col in ('peak_allocated_bytes', 'peak_reserved_bytes', 'peak_increment_over_model_bytes'):    df[col + '_GiB'] = df[col] / (1024 ** 3)display(df)summary = {    'head': HEAD,    'preflight_timestamp_utc': pre.get('timestamp_utc'),    'cases': rows,    'training_started': False,    'optimizer_steps': 0,    'datasets_loaded': False,    'test_split_touched': False,}summary_path = OUT / 'gpu_memory_audit_summary.json'summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False))print('Informe:', summary_path)print('TRAINING_STARTED: False')print('DATASETS_LOADED: False')print('TEST_SPLIT_TOUCHED: False')print('OPTIMIZER_STEPS: 0')